# Feasibility-based bound tightening (FBBT)

`pounce` can run interval-arithmetic propagation through each
nonlinear constraint's expression DAG to tighten variable bounds
before the IPM starts. The classic example: a circle constraint
`x² + y² ≤ 1` with a wide user-declared box `x, y ∈ [-100, 100]`.
FBBT shrinks the box to `[-1, 1]` for free.

This is off by default — issue [#62](https://github.com/jkitchin/pounce/issues/62)
deliberately defers the default flip until benchmark evidence
justifies it. This notebook walks through enabling it via Pyomo.

See the [FBBT reference](../../docs/src/fbbt.md) for the algorithm,
operator support, and soundness guarantees.

In [1]:
import io
import contextlib
import pyomo_pounce  # noqa: F401 -- registers 'pounce' with SolverFactory
import pyomo.environ as pyo


## A problem with loose bounds

Two-variable optimization on a circle, with the box declared
100× larger than necessary:

$$
\min_{x, y}\; (x - 0.5)^2 + (y - 0.5)^2
\quad\text{s.t.}\quad x^2 + y^2 = 1,\; -100 \le x \le 100,\; -100 \le y \le 100.
$$

The optimum is the closest point on the unit circle to `(0.5, 0.5)` —
i.e. `(√2/2, √2/2) ≈ (0.707, 0.707)`. But the IPM doesn't know
about the circle constraint until it evaluates it; the loose `[-100,
100]` box dominates its initial step-size heuristics and slows
convergence.

In [2]:
def build_model():
    m = pyo.ConcreteModel()
    m.x = pyo.Var(bounds=(-100.0, 100.0), initialize=0.0)
    m.y = pyo.Var(bounds=(-100.0, 100.0), initialize=0.0)
    m.obj = pyo.Objective(expr=(m.x - 0.5)**2 + (m.y - 0.5)**2)
    m.circle = pyo.Constraint(expr=m.x**2 + m.y**2 == 1.0)
    return m


def solve(extra_options=None):
    """Solve once with `extra_options` merged onto a quiet baseline.
    Returns (n_iters, x*, y*, fbbt_report_line_or_None)."""
    m = build_model()
    solver = pyo.SolverFactory("pounce")
    solver.options["print_level"] = 5  # so we get the iter-count line
    solver.options["tol"] = 1e-8
    for k, v in (extra_options or {}).items():
        solver.options[k] = v
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        solver.solve(m, tee=True)
    log = buf.getvalue()
    iters = None
    fbbt_line = None
    for line in log.splitlines():
        if line.startswith("Number of Iterations"):
            iters = int(line.split(":")[1].strip())
        if line.startswith("Presolve FBBT:"):
            fbbt_line = line
    return iters, pyo.value(m.x), pyo.value(m.y), fbbt_line


## Run 1 — solve as-is (no presolve, no FBBT)

In [3]:
iters, x, y, fbbt = solve()
print(f"iters : {iters}")
print(f"x*    : {x:.6f}")
print(f"y*    : {y:.6f}")
print(f"|x|+|y| residual to circle: {abs(x*x + y*y - 1.0):.2e}")
print(f"FBBT report: {fbbt or 'none'}")

iters : 7
x*    : 0.707107
y*    : 0.707107
|x|+|y| residual to circle: 2.22e-16
FBBT report: none


## Run 2 — `presolve=yes` but no FBBT

Pounce's existing linear presolve doesn't do anything useful here:
the only constraint is nonlinear (it's a circle), so Phase 1's
linear bound-tightening has nothing to propagate.

In [4]:
iters, x, y, fbbt = solve({"presolve": "yes"})
print(f"iters : {iters}")
print(f"x*    : {x:.6f}, y* : {y:.6f}")
print(f"FBBT report: {fbbt or 'none'}")

iters : 7
x*    : 0.707107, y* : 0.707107
FBBT report: none


## Run 3 — turn on FBBT

`presolve=yes presolve_fbbt=yes`: now the circle constraint
propagates into the variable box. After one forward+reverse pass,
the box becomes a subset of `[-1, 1]²`.

In [5]:
iters, x, y, fbbt = solve({"presolve": "yes", "presolve_fbbt": "yes"})
print(f"iters : {iters}")
print(f"x*    : {x:.6f}, y* : {y:.6f}")
print(f"FBBT report: {fbbt}")

iters : 7
x*    : 0.707107, y* : 0.707107
FBBT report: Presolve FBBT: 2 sweeps, 2 variable tightenings (Σ|Δ|=1.980e2)


## A more striking case: `exp(x) ≤ 10`

The user-declared box says `x ∈ [-50, 50]`, but the constraint
`exp(x) ≤ 10` forces `x ≤ ln 10 ≈ 2.303`. Without FBBT the IPM
has to discover this via line search. With FBBT it's known before
the first iteration.

In [6]:
def build_exp_model():
    m = pyo.ConcreteModel()
    m.x = pyo.Var(bounds=(-50.0, 50.0), initialize=0.0)
    m.obj = pyo.Objective(expr=(m.x - 5.0) ** 2)  # wants x = 5; constraint blocks it
    m.c = pyo.Constraint(expr=pyo.exp(m.x) <= 10.0)
    return m


def solve_exp(extra_options=None):
    m = build_exp_model()
    solver = pyo.SolverFactory("pounce")
    solver.options["print_level"] = 5
    solver.options["tol"] = 1e-8
    for k, v in (extra_options or {}).items():
        solver.options[k] = v
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        solver.solve(m, tee=True)
    log = buf.getvalue()
    iters = None
    fbbt = None
    for line in log.splitlines():
        if line.startswith("Number of Iterations"):
            iters = int(line.split(":")[1].strip())
        if line.startswith("Presolve FBBT:"):
            fbbt = line
    return iters, pyo.value(m.x), fbbt


for name, opts in [
    ("no presolve",   {}),
    ("presolve only", {"presolve": "yes"}),
    ("presolve+FBBT", {"presolve": "yes", "presolve_fbbt": "yes"}),
]:
    iters, x, fbbt = solve_exp(opts)
    print(f"{name:>16s}: iters={iters}, x*={x:.6f}  {fbbt or ''}")

     no presolve: iters=8, x*=2.302585  
   presolve only: iters=8, x*=2.302585  
   presolve+FBBT: iters=9, x*=2.302585  Presolve FBBT: 2 sweeps, 1 variable tightenings (Σ|Δ|=4.770e1)


## What the report fields mean

```
Presolve FBBT: 1 sweeps, 4 variable tightenings (Σ|Δ|=1.98e2)
```

* `sweeps` — number of outer iterations actually executed
  (≤ `fbbt_max_iter`, default 10).
* `variable tightenings` — total count of per-variable `(x_lo[j],
  x_hi[j])` updates that strictly improved the box.
* `Σ|Δ|` — sum of absolute bound improvements across all updates.

If FBBT detects infeasibility, you'll see an additional line:
`pounce: FBBT detected infeasibility (witness constraint N)`.

## Why FBBT is off by default

Look at the iteration counts above: FBBT *successfully* tightened
the bounds in both runs (the `Σ|Δ|` field in the FBBT report shows
that real work happened — 198 units of total bound shrinkage on
the circle problem, 47.7 units on the exp problem). But on these
small problems the IPM iteration count doesn't go down. Sometimes
it goes up.

This is exactly the situation the issue
[#62](https://github.com/jkitchin/pounce/issues/62) calls out:
tighter bounds can change the IPM's starting point, line search,
and warm-start behavior in ways that don't always reduce work.
Whether FBBT pays off on *your* problem depends on (a) how loose
your bounds are vs. the constraint geometry, (b) how badly the IPM
would otherwise stumble on the loose box, and (c) whether the
additional row drops + LICQ promotion that FBBT enables
downstream pay off in the rest of the presolve pipeline.

Try it on a problem you care about. If iteration count drops or
your LICQ verdict goes from `StructuralRank` to `Full`, keep it
on. Otherwise, leave it off.

## Tunable knobs

| Option | Default | Effect |
|---|---|---|
| `presolve_fbbt` | `no` | Master switch. Requires `presolve=yes`. |
| `fbbt_tol` | `1e-6` | Minimum per-variable bound improvement to keep iterating. |
| `fbbt_max_iter` | `10` | Outer-sweep cap. |
| `fbbt_max_constraints` | `0` | Per-sweep cap on constraints inspected (`0` = unlimited). |

## What FBBT can't see

Only TNLPs that expose constraint expression trees benefit from
FBBT. Today that's `.nl`-loaded problems via `NlTnlp` — which is
what Pyomo writes (and what this notebook used). Problems loaded
through `pounce.Problem` from Python class definitions or via the
C callback bridge currently opt out silently: the option is
accepted but produces no `Presolve FBBT:` line.

Adding FBBT support to another problem source means implementing
`pounce_nlp::expression_provider::ExpressionProvider` for it. See
the [FBBT reference](../../docs/src/fbbt.md) for the recipe.